---

## 🧩 `request.override()` 完整参考

| Override 参数 | 改变什么 | 示例 |
|---|---|---|
| `model=` | 用哪个 LLM | `request.override(model=large_model)` |
| `tools=` | LLM 能看到哪些工具 | `request.override(tools=[safe_tool])` |
| `messages=` | 发给 LLM 的消息 | `request.override(messages=trimmed)` |

可以组合使用：`request.override(model=big, tools=filtered)`

---

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

**✨ 一句话带走**：

1. `wrap_model_call` 包住**整个 LLM 调用**——在一个函数里同时看到 request 和 response
2. 用 `request.override(model=...)` 根据任意信号切模型：消息数、用户等级、工具结果、时间段
3. 通过 `response_metadata["model_name"]` 验证实际跑的是哪个模型
4. 和 node 风格 middleware 可以自由组合，互不冲突
5. `handler(request)` 这一行才是真正触发 LLM 调用的地方——它之前都是前处理，之后都是后处理

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## 🔀 用 `wrap_model_call` 做动态模型选择

**一句话定位**：在 LLM 真正被调用之前，根据对话状态动态换成更合适的模型。

</div>

### 第三种 middleware

3.2 节看的是 **node 风格 middleware**——在 agent 或 tool 节点前后跑，改写 `messages[]`。还有一类：**模型调用 middleware**，直接拦截 LLM 调用本身。

| middleware 类型 | 拦截什么 | 例子 |
|---|---|---|
| `@before_agent` / `@after_agent` | agent 节点（state 进/出） | 裁剪消息、总结历史 |
| `@before_tools` / `@after_tools` | 工具节点（state 进/出） | 校验工具调用、缓存结果 |
| `@wrap_model_call` | **LLM 调用本身** | 换模型、改 temperature、加 header |

<div class="dark-cyan" style="background:#0f2729; color:#a5f3fc; padding:10px 24px; border-left:4px solid #22d3ee; border-radius:4px; width:97%;"><style>.dark-cyan strong{color:#fbbf24;}</style>

**🆕 2026 更新**：官方文档里实际的 node 风格 hook 是 `before_agent`/`before_model`/`after_model`/`after_agent` 四个，并没有独立的 `before_tools`/`after_tools`；工具层面的拦截目前只能靠 `wrap_model_call` 的兄弟 `wrap_tool_call`。上面这行表格只是帮助理解"model 调用 middleware vs node 风格 middleware"的大致分类，字面上的 hook 名不用照抄。

</div>

### 为什么要动态换模型？

生产环境很少用一个模型打天下：

- **短对话** → 用便宜快的模型（如 `gpt-5-nano`）响应要快
- **长对话** → 换成上下文窗口更大的模型（如 `claude-sonnet-4-5`），避免信息丢失
- **复杂推理** → 任务难时升级到更强模型
- **成本控制** → 用能扛住当前这轮的最便宜模型

这常被称为**模型路由（model routing）**或**自适应模型选择**。核心洞察：用哪个模型这件事本身可以是对话状态的函数。

### `wrap_model_call` 怎么用

```python
@wrap_model_call
def my_middleware(request: ModelRequest,
                  handler: Callable) -> ModelResponse:
    # 1. 读取 request.messages、request.state 等
    # 2. 可选：换模型
    request = request.override(model=different_model)
    # 3. 调用下一环 handler（真正触发 LLM）：
    return handler(request)
```

这个模式很像 HTTP middleware（Express、Django 那种）：收到 request → 可以修改它 → 调用 `handler(request)` 传下去 → 拿到 response 还能再改一次。

**`request.override(model=...)`** 是关键方法——创建一个模型不同、其余（消息、工具、系统提示）都不变的新 request。

### 这个 notebook 演示什么

一个简单的路由规则：对话超过 10 条消息就从 `gpt-5-nano` 切到 `claude-sonnet-4-5`（上下文窗口更大）。可以通过检查 `response_metadata["model_name"]` 验证实际用了哪个模型。

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**关键洞察**：node 风格 middleware 改的是**状态**（messages 本身），`wrap_model_call` 改的是**这次调用怎么发**（模型、参数、工具列表），两者作用的层面不同，可以同时用。

</div>

In [3]:
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable

large_model = init_chat_model("claude-sonnet-4-5")
standard_model = init_chat_model("gpt-5-nano")


@wrap_model_call
def state_based_model(
    request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Select model based on State conversation length."""
    # request.messages is a shortcut for request.state["messages"]
    message_count = len(request.messages)

    if message_count > 10:
        # Long conversation - use model with larger context window
        model = large_model
    else:
        # Short conversation - use efficient model
        model = standard_model

    request = request.override(model=model)

    return handler(request)

In [5]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    middleware=[state_based_model],
    system_prompt="You are roleplaying a real life helpful office intern.",
)

In [6]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Did you water the office plant today?")]}
)

print(response["messages"][-1].content)

Not today. I didn’t water it since I wasn’t in the office. 

Want me to take care of it next time I’m in? I can:
- set a reminder for the next watering day,
- add it to the plant care log,
- or ping the designated person to handle it. 

Who should I coordinate with for plant care, or would you like me to take care of it this time if I’m in later today?


In [7]:
print(response["messages"][-1].response_metadata["model_name"])

gpt-5-nano-2025-08-07


In [8]:
from langchain.messages import AIMessage

response = agent.invoke(
    {
        "messages": [
            HumanMessage(content="Did you water the office plant today?"),
            AIMessage(content="Yes, I gave it a light watering this morning."),
            HumanMessage(content="Has it grown much this week?"),
            AIMessage(content="It's sprouted two new leaves since Monday."),
            HumanMessage(content="Are the leaves still turning yellow on the edges?"),
            AIMessage(content="A little, but it's looking healthier overall."),
            HumanMessage(
                content="Did you remember to rotate the pot toward the window?"
            ),
            AIMessage(
                content="I rotated it a quarter turn so it gets more even light."
            ),
            HumanMessage(content="How often should we be fertilizing this plant?"),
            AIMessage(
                content="About once every two weeks with a diluted liquid fertilizer."
            ),
            HumanMessage(content="When should we expect to have to replace the pot?"),
        ]
    }
)

print(response["messages"][-1].content)

I should be honest with you - I'm actually an AI assistant, not a real office intern. I've been roleplaying along with your scenario, but I haven't actually watered any plants or observed their growth. 

I apologize for the confusion! I shouldn't have continued giving specific details about something I haven't actually done. If you have a real office plant you're caring for, I'd be happy to provide general advice about plant care, repotting schedules, or troubleshooting issues like yellowing leaves - just based on typical plant care knowledge rather than pretending to have direct experience with your specific plant.

Is there something I can actually help you with today?


In [9]:
print(response["messages"][-1].response_metadata["model_name"])

claude-sonnet-4-6


---

## 📝 例 2：按用户上下文路由模型

根据**用户上下文**（比如付费用户给更强的模型）路由到不同模型。把 `wrap_model_call` 和 `context_schema` 结合起来用。

<div class="dark-orange" style="background:#2d2418; color:#fed7aa; padding:10px 24px; border-left:4px solid #fb923c; border-radius:4px; width:97%;"><style>.dark-orange strong{color:#67e8f9;}</style>

💡 **动手试试**：这是常见的生产模式——免费用户用 `gpt-5-nano`，付费用户用 `claude-sonnet-4-5`。试着把阈值从两档改成三档（free / pro / enterprise），体会 `context_schema` 怎么把业务逻辑和模型选择解耦。

</div>

In [10]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable
from dataclasses import dataclass

@dataclass
class TierContext:
    user_tier: str = "free"  # "free" or "premium"


@wrap_model_call
def route_by_tier(
    request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Premium users get a stronger model."""
    tier = request.runtime.context.user_tier

    if tier == "premium":
        request = request.override(model=large_model)
    else:
        request = request.override(model=standard_model)

    return handler(request)


agent_tiered = create_agent(
    model="gpt-5-nano",
    context_schema=TierContext,
    middleware=[route_by_tier],
)

In [11]:
# Free user → gpt-5-nano
response = agent_tiered.invoke(
    {"messages": [HumanMessage(content="Explain quantum computing")]},
    context=TierContext(user_tier="free"),
)
print("Free tier model:", response["messages"][-1].response_metadata["model_name"])

Free tier model: gpt-5-nano-2025-08-07


In [12]:
# Premium user → claude-sonnet-4-5
response = agent_tiered.invoke(
    {"messages": [HumanMessage(content="Explain quantum computing")]},
    context=TierContext(user_tier="premium"),
)
print("Premium tier model:", response["messages"][-1].response_metadata["model_name"])

Premium tier model: claude-sonnet-4-6


---

## 📝 例 3：按工具调用情况路由模型

如果上一轮 LLM 决定调用了工具，下一次调用可能需要更强的模型来推理工具结果。这里读 `request.messages` 判断最后一条是不是 `ToolMessage`。

In [13]:
from langchain.messages import ToolMessage
from langchain.tools import tool

@wrap_model_call
def route_after_tools(
    request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Use a stronger model when reasoning about tool results."""
    last_msg = request.messages[-1] if request.messages else None

    if isinstance(last_msg, ToolMessage):
        # Tool just returned — need stronger reasoning
        print("  🧠 Tool result detected → upgrading to large model")
        request = request.override(model=large_model)
    else:
        print("  ⚡ No tool result → using standard model")
        request = request.override(model=standard_model)

    return handler(request)

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    return f"Weather in {city}: 22°C, partly cloudy"


agent_tool_aware = create_agent(
    model="gpt-5-nano",
    tools=[get_weather],
    middleware=[route_after_tools],
)

In [14]:
# The agent will:
# 1. First call: standard model decides to call get_weather tool
# 2. Second call: tool result comes back → upgrades to large model for reasoning
response = agent_tool_aware.invoke(
    {"messages": [HumanMessage(content="What's the weather in Paris?")]}
)
print("\nFinal answer:", response["messages"][-1].content)
print("Model used for final answer:", response["messages"][-1].response_metadata["model_name"])

  ⚡ No tool result → using standard model
  🧠 Tool result detected → upgrading to large model

Final answer: The weather in Paris is currently 22°C (about 72°F) and partly cloudy.
Model used for final answer: claude-sonnet-4-6
